In [1]:
# ============================================================
# CELL 1 — Install dependencies
# ============================================================
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

pip_install('torch', 'torchvision', '--index-url', 'https://download.pytorch.org/whl/cu124')

print(" Dependencies ready.")

 Dependencies ready.


In [2]:
import os
for p in os.listdir('/kaggle/input/datasets/garymk/kitti-3d-object-detection-dataset'):
    print(p)

kitti_dbinfos_train.pkl
training
kitti_infos_train.pkl
testing
kitti_infos_test.pkl
kitti_infos_val.pkl
gt_database
devkit_object
kitti_infos_trainval.pkl


In [3]:
# ============================================================
# CELL 2 — KITTI Dataset: Mount from Kaggle Datasets
# ============================================================
# The KITTI 3D Object Detection dataset is available on Kaggle at:
#   https://www.kaggle.com/datasets/garymk/kitti-3d-object-detection-dataset
#
# HOW TO ADD IT TO YOUR SESSION (no download needed):
#   1. Open your Kaggle Notebook
#   2. Click the "+" icon next to "Input" on the right panel
#   3. Search "KITTI 3D Object Detection" → Add dataset by garymk
#   4. The data will be mounted at /kaggle/input/kitti-3d-object-detection-dataset/
#
# Expected structure after mounting:
#   /kaggle/input/kitti-3d-object-detection-dataset/
#   └── training/
#       ├── velodyne/     ← .bin files
#       ├── label_2/      ← .txt labels
#       └── calib/        ← .txt calibration
#
# ImageSets (train.txt / val.txt) need to be created if not present:
import os
from pathlib import Path
import urllib.request
import shutil


KITTI_ROOT   = '/kaggle/input/datasets/garymk/kitti-3d-object-detection-dataset'
IMAGESET_DIR = '/kaggle/input/datasets/codingyodha/imagesets'
SAVE_DIR     = '/kaggle/working/checkpoints'
CHECKPOINT_INPUT = '/kaggle/input/datasets/codingyodha/session-1-checkpoints'

os.makedirs(IMAGESET_DIR, exist_ok=True)
os.makedirs(SAVE_DIR,     exist_ok=True)



# Copy all checkpoints from mounted dataset back to working dir
if os.path.exists(CHECKPOINT_INPUT):
    for f in os.listdir(CHECKPOINT_INPUT):
        src = os.path.join(CHECKPOINT_INPUT, f)
        dst = os.path.join(SAVE_DIR, f)
        shutil.copy2(src, dst)
        print(f"  Restored: {f}")
    print(f"Checkpoints restored to {SAVE_DIR}")
else:
    print("No checkpoint dataset found — starting fresh")
# # Official splits hosted by multiple repos — these are canonical
# train_url = 'https://raw.githubusercontent.com/traveller59/second.pytorch/master/second/data/ImageSets/train.txt'
# val_url   = 'https://raw.githubusercontent.com/traveller59/second.pytorch/master/second/data/ImageSets/val.txt'

# urllib.request.urlretrieve(train_url, os.path.join(IMAGESET_DIR, 'train.txt'))
# urllib.request.urlretrieve(val_url,   os.path.join(IMAGESET_DIR, 'val.txt'))

# with open(os.path.join(IMAGESET_DIR, 'train.txt')) as f:
#     n_train = sum(1 for l in f if l.strip())
# with open(os.path.join(IMAGESET_DIR, 'val.txt')) as f:
#     n_val = sum(1 for l in f if l.strip())

# print(f"Official split loaded: {n_train} train / {n_val} val")
# ── Auto-generate ImageSets from available .bin files if not present ──────
train_txt = os.path.join(IMAGESET_DIR, 'train.txt')
val_txt   = os.path.join(IMAGESET_DIR, 'val.txt')

if not os.path.exists(train_txt) or not os.path.exists(val_txt):
    velodyne_dir = os.path.join(KITTI_ROOT, 'training', 'velodyne')
    all_ids = sorted([
        f.replace('.bin', '')
        for f in os.listdir(velodyne_dir) if f.endswith('.bin')
    ])
    split_idx = int(len(all_ids) * (6/7))   # ~87.5% train, ~12.5% val (KITTI standard ratio)
    train_ids = all_ids[:split_idx]
    val_ids   = all_ids[split_idx:]

    with open(train_txt, 'w') as f:
        f.write('\n'.join(train_ids))
    with open(val_txt, 'w') as f:
        f.write('\n'.join(val_ids))

    print(f" ImageSets created: {len(train_ids)} train / {len(val_ids)} val frames")
else:
    with open(train_txt) as f:
        n_train = sum(1 for l in f if l.strip())
    with open(val_txt) as f:
        n_val = sum(1 for l in f if l.strip())
    print(f" ImageSets found: {n_train} train / {n_val} val frames")

print(f"KITTI root : {KITTI_ROOT}")
print(f"Save dir   : {SAVE_DIR}")

  Restored: train_metrics_session_backup.csv
  Restored: session_checkpoint.pth
  Restored: best.pth
  Restored: ckpt_epoch0052_session_end.pth
  Restored: train_metrics.csv
Checkpoints restored to /kaggle/working/checkpoints
 ImageSets found: 3712 train / 3769 val frames
KITTI root : /kaggle/input/datasets/garymk/kitti-3d-object-detection-dataset
Save dir   : /kaggle/working/checkpoints


In [4]:
# ============================================================
# CELL 3 — dataset.py  (KITTIPillarDataset + pillar encoding)
# ============================================================
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.utils.data.distributed import DistributedSampler
from pathlib import Path
from typing import Optional

# ---------------------------------------------------------------------------
# KITTI class map
# ---------------------------------------------------------------------------
KITTI_CLASSES = {'Car': 0, 'Pedestrian': 1, 'Cyclist': 2}

# ---------------------------------------------------------------------------
# BEV grid configuration
# ---------------------------------------------------------------------------
BEV_CFG = dict(
    x_min=-39.68, x_max=39.68,
    y_min=-39.68, y_max=39.68,
    z_min=-3.0,   z_max=1.0,
    pillar_size_x=0.16,
    pillar_size_y=0.16,
    max_points_per_pillar=32,
    max_pillars=16000,
    bev_h=512,
    bev_w=512,
)

# FIX [Moderate — BN stability]:
# Minimum number of distinct pillars returned per sample.
# If fewer real pillars are found, we cyclically duplicate existing pillars
# to reach this count. Duplicates share coords with their source, so
# BEVScatter's dedup removes them before scatter — no spurious features.
# This ensures BatchNorm1d in PillarFeatureNet always sees enough samples.
MIN_PILLARS = 8


def points_to_pillars(points: np.ndarray, cfg: dict):
    """
    Convert raw LiDAR point cloud to pillar tensors (vectorised NumPy).

    Args:
        points : (N, 4)  float32   x, y, z, intensity
        cfg    : BEV_CFG dict

    Returns:
        pillars      : (P, max_pts, 4)  float32  (zero-padded)
        num_points   : (P,)             int32
        coords       : (P, 2)           int32    (row, col)
        num_pillars  : int
    """
    x_min, x_max = cfg['x_min'], cfg['x_max']
    y_min, y_max = cfg['y_min'], cfg['y_max']
    z_min, z_max = cfg['z_min'], cfg['z_max']
    ps_x,  ps_y  = cfg['pillar_size_x'], cfg['pillar_size_y']
    max_pts       = cfg['max_points_per_pillar']
    max_pillars   = cfg['max_pillars']
    H, W          = cfg['bev_h'], cfg['bev_w']

    # --- Filter to valid range ---
    mask = (
        (points[:, 0] >= x_min) & (points[:, 0] < x_max) &
        (points[:, 1] >= y_min) & (points[:, 1] < y_max) &
        (points[:, 2] >= z_min) & (points[:, 2] < z_max)
    )
    points = points[mask]

    if len(points) == 0:
        empty_pillars = np.zeros((MIN_PILLARS, max_pts, 4), dtype=np.float32)
        empty_numpts  = np.ones((MIN_PILLARS,),              dtype=np.int32)
        empty_coords  = np.zeros((MIN_PILLARS, 2),           dtype=np.int32)
        return empty_pillars, empty_numpts, empty_coords, MIN_PILLARS

    # --- Assign each point to a (col, row) pillar ---
    col_idx   = ((points[:, 0] - x_min) / ps_x).astype(np.int32).clip(0, W - 1)
    row_idx   = ((points[:, 1] - y_min) / ps_y).astype(np.int32).clip(0, H - 1)
    pillar_id = row_idx * W + col_idx

    sort_order  = np.argsort(pillar_id, kind='stable')
    sorted_pts  = points[sort_order]
    sorted_ids  = pillar_id[sort_order]

    unique_ids, first_occ, counts = np.unique(
        sorted_ids, return_index=True, return_counts=True)

    if len(unique_ids) > max_pillars:
        top_k      = np.argpartition(counts, -max_pillars)[-max_pillars:]
        unique_ids = unique_ids[top_k]
        first_occ  = first_occ[top_k]
        counts     = counts[top_k]

    P      = len(unique_ids)
    n_take = np.minimum(counts, max_pts)

    col_range   = np.arange(max_pts, dtype=np.int32)[None, :]
    gather_idx  = first_occ[:, None] + np.minimum(
        col_range, (n_take - 1)[:, None])
    pillars_raw = sorted_pts[gather_idx]

    valid_mask  = col_range < n_take[:, None]
    pillars     = pillars_raw * valid_mask[:, :, None].astype(np.float32)
    num_points  = n_take.astype(np.int32)
    coords      = np.stack(
        [unique_ids // W, unique_ids % W], axis=1).astype(np.int32)

    # FIX: Minimum pillar guarantee for BN stability
    if P < MIN_PILLARS:
        n_dup      = MIN_PILLARS - P
        dup_idx    = np.arange(n_dup) % P
        pillars    = np.concatenate([pillars,    pillars[dup_idx]],    axis=0)
        num_points = np.concatenate([num_points, num_points[dup_idx]], axis=0)
        coords     = np.concatenate([coords,     coords[dup_idx]],     axis=0)
        P          = MIN_PILLARS

    return pillars, num_points, coords, P


def parse_kitti_calib(calib_path: str) -> np.ndarray:
    """Parse KITTI calibration file and return Tr_velo_to_cam (3×4)."""
    with open(calib_path, 'r') as f:
        for line in f:
            if line.startswith('Tr_velo_to_cam'):
                vals = list(map(float, line.strip().split(':')[1].split()))
                return np.array(vals, dtype=np.float64).reshape(3, 4)
    raise ValueError(f'Tr_velo_to_cam not found in {calib_path}')


def cam_to_lidar(loc_cam: np.ndarray, ry: float,
                 velo_to_cam: np.ndarray):
    """
    Convert a single KITTI GT box from camera frame to LiDAR frame.
    yaw_lidar = -ry - π/2  (standard KITTI → LiDAR convention)
    """
    R = velo_to_cam[:, :3]
    t = velo_to_cam[:, 3]
    p_lidar = R.T @ (loc_cam - t)
    lx, ly, lz = p_lidar[0], p_lidar[1], p_lidar[2]
    yaw_lidar = -ry - np.pi / 2
    yaw_lidar = (yaw_lidar + np.pi) % (2 * np.pi) - np.pi
    return float(lx), float(ly), float(lz), float(yaw_lidar)


def parse_kitti_label(label_path: str, calib_path: str):
    """
    Parse one KITTI label file and convert boxes to LiDAR frame.
    Returns list of dicts: class_id, lx, ly, lz, w, l, h, yaw
    """
    velo_to_cam = parse_kitti_calib(calib_path)
    objs = []
    with open(label_path, 'r') as f:
        for line in f:
            parts   = line.strip().split()
            cls_str = parts[0]
            if cls_str not in KITTI_CLASSES:
                continue
            h, w, l = float(parts[8]),  float(parts[9]),  float(parts[10])
            x, y, z = float(parts[11]), float(parts[12]), float(parts[13])
            ry      = float(parts[14])
            lx, ly, lz, yaw_lidar = cam_to_lidar(
                np.array([x, y, z], dtype=np.float64), ry, velo_to_cam)
            objs.append(dict(
                class_id=KITTI_CLASSES[cls_str],
                lx=lx, ly=ly, lz=lz,
                w=w, l=l, h=h,
                yaw=yaw_lidar,
            ))
    return objs


class KITTIPillarDataset(Dataset):
    """
    PyTorch Dataset for KITTI 3D Object Detection (LiDAR pillar format).

    Each __getitem__ returns:
        pillars      : (P, max_pts, 4)  float32
        num_points   : (P,)             int32
        coords       : (P, 2)           int32    row, col
        num_pillars  : int
        boxes        : (M, 8)  float32  [class_id, x, y, z, w, l, h, yaw]
        frame_id     : str
    """
    def __init__(self, root: str, split: str = 'train',
                 imageset_path: Optional[str] = None,
                 cfg: dict = BEV_CFG):
        self.root  = Path(root)
        self.cfg   = cfg
        self.split = split

        if imageset_path is None:
            imageset_path = self.root.parent / 'ImageSets' / f'{split}.txt'
        with open(imageset_path, 'r') as f:
            self.frame_ids = [l.strip() for l in f if l.strip()]

        self.lidar_dir = self.root / 'training' / 'velodyne'
        self.label_dir = self.root / 'training' / 'label_2'
        self.calib_dir = self.root / 'training' / 'calib'

    def __len__(self):
        return len(self.frame_ids)

    def __getitem__(self, idx: int) -> dict:
        fid      = self.frame_ids[idx]
        bin_path = self.lidar_dir / f'{fid}.bin'
        points   = np.fromfile(str(bin_path), dtype=np.float32).reshape(-1, 4)

        boxes = np.zeros((0, 8), dtype=np.float32)
        label_path = self.label_dir / f'{fid}.txt'
        calib_path = self.calib_dir / f'{fid}.txt'
        if label_path.exists() and calib_path.exists():
            objs = parse_kitti_label(str(label_path), str(calib_path))
            if objs:
                boxes = np.array([[
                    o['class_id'], o['lx'], o['ly'], o['lz'],
                    o['w'], o['l'], o['h'], o['yaw']
                ] for o in objs], dtype=np.float32)

        # --- 3D Data Augmentation (Training ONLY) ---
        if self.split == 'train' and len(boxes) > 0:
            # 1. Random Flip Y (Left/Right)
            if np.random.rand() < 0.5:
                points[:, 1] = -points[:, 1]
                boxes[:, 2]  = -boxes[:, 2]
                boxes[:, 7]  = -boxes[:, 7]
            # 2. Random Flip X (Front/Back)
            if np.random.rand() < 0.5:
                points[:, 0] = -points[:, 0]
                boxes[:, 1]  = -boxes[:, 1]
                boxes[:, 7]  = np.pi - boxes[:, 7]
            # 3. Random Global Rotation (-45 to +45 degrees)
            if np.random.rand() < 0.5:
                rot_angle = np.random.uniform(-np.pi/4, np.pi/4)
                cos_val, sin_val = np.cos(rot_angle), np.sin(rot_angle)
                rot_mat = np.array([[cos_val, -sin_val],
                                    [sin_val,  cos_val]], dtype=np.float32)
                points[:, :2] = points[:, :2] @ rot_mat
                boxes[:, 1:3] = boxes[:, 1:3] @ rot_mat
                boxes[:, 7]  += rot_angle

        pillars, num_pts, coords, n_pil = points_to_pillars(points, self.cfg)

        return {
            'pillars'    : torch.from_numpy(pillars),
            'num_points' : torch.from_numpy(num_pts),
            'coords'     : torch.from_numpy(coords),
            'num_pillars': n_pil,
            'boxes'      : torch.from_numpy(boxes),
            'frame_id'   : fid,
        }


def kitti_collate_fn(batch: list) -> dict:
    """Concatenate pillars across batch, prepend batch_idx to coords."""
    batch_pillars = []
    batch_numpts  = []
    batch_coords  = []
    batch_boxes   = []
    frame_ids     = []

    for b_idx, sample in enumerate(batch):
        P = sample['num_pillars']
        batch_pillars.append(sample['pillars'])
        batch_numpts.append(sample['num_points'])
        b_col = torch.full((P, 1), b_idx, dtype=torch.int32)
        batch_coords.append(torch.cat([b_col, sample['coords']], dim=1))
        batch_boxes.append(sample['boxes'])
        frame_ids.append(sample['frame_id'])

    return {
        'pillars'    : torch.cat(batch_pillars, dim=0),
        'num_points' : torch.cat(batch_numpts,  dim=0),
        'coords'     : torch.cat(batch_coords,  dim=0),
        'boxes'      : batch_boxes,
        'batch_size' : len(batch),
        'frame_ids'  : frame_ids,
    }


def build_dataloaders(kitti_root: str,
                      imageset_dir: str,
                      batch_size: int,
                      num_workers: int,
                      rank: int,
                      world_size: int,
                      cfg: dict = BEV_CFG):
    """Build train/val DataLoaders with DistributedSampler."""
    train_ds = KITTIPillarDataset(
        kitti_root, 'train',
        os.path.join(imageset_dir, 'train.txt'), cfg)
    val_ds   = KITTIPillarDataset(
        kitti_root, 'val',
        os.path.join(imageset_dir, 'val.txt'), cfg)

    train_sampler = DistributedSampler(
        train_ds, num_replicas=world_size, rank=rank,
        shuffle=True, drop_last=True)
    val_sampler   = DistributedSampler(
        val_ds, num_replicas=world_size, rank=rank,
        shuffle=False, drop_last=False)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size,
        sampler=train_sampler,
        num_workers=num_workers,
        collate_fn=kitti_collate_fn,
        pin_memory=True,
        persistent_workers=(num_workers > 0))
    val_loader   = DataLoader(
        val_ds, batch_size=batch_size,
        sampler=val_sampler,
        num_workers=num_workers,
        collate_fn=kitti_collate_fn,
        pin_memory=True,
        persistent_workers=(num_workers > 0))

    return train_loader, val_loader, train_sampler


print(" dataset.py cell loaded.")

 dataset.py cell loaded.


In [5]:
# ============================================================
# CELL 4 — model.py  (BEVWaveFormer architecture)
# ============================================================
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
EPS = 1e-6
_SINC_TAYLOR_THRESH = 0.05
_SPECTRAL_CLAMP = 50.0


# ===========================================================================
# 1.  Pillar Feature Encoder
# ===========================================================================
class PillarFeatureNet(nn.Module):
    """
    Lightweight VFE for cylindrical pillars (PointPillars-style).
    Augments each point with pillar-center offsets (Δx, Δy) and distance.
    BN eps=1e-3 prevents 1/σ explosion on sparse (mostly zero-padded) pillars.
    """
    def __init__(self, in_channels: int = 4, out_channels: int = 64,
                 max_points: int = 32):
        super().__init__()
        augmented_in = in_channels + 3   # +Δx, +Δy, +dist
        self.mlp = nn.Sequential(
            nn.Linear(augmented_in, 32, bias=False),
            nn.BatchNorm1d(32, eps=1e-3),
            nn.SiLU(inplace=True),
            nn.Linear(32, out_channels, bias=False),
            nn.BatchNorm1d(out_channels, eps=1e-3),
            nn.SiLU(inplace=True),
        )
        self.out_channels = out_channels
        self.max_points   = max_points

    def forward(self, pillars: torch.Tensor,
                num_points_per_pillar: torch.Tensor) -> torch.Tensor:
        P, T, C = pillars.shape
        mask    = (torch.arange(T, device=pillars.device).unsqueeze(0)
                   < num_points_per_pillar.unsqueeze(1))
        mask_f  = mask.unsqueeze(-1).float()

        valid_cnt = num_points_per_pillar.clamp(min=1).float().view(P, 1, 1)
        center_xy = (pillars[..., :2] * mask_f).sum(dim=1, keepdim=True) / valid_cnt
        delta_xy  = pillars[..., :2] - center_xy
        dist      = torch.norm(pillars[..., :3], dim=-1, keepdim=True)

        x = torch.cat([pillars, delta_xy, dist], dim=-1)
        x = x * mask_f

        x_flat = x.view(P * T, -1)
        feats  = self.mlp(x_flat)
        feats  = feats.view(P, T, self.out_channels) * mask_f
        pillar_feats, _ = feats.max(dim=1)
        return pillar_feats


# ===========================================================================
# 2.  BEV Scatter
# ===========================================================================
class BEVScatter(nn.Module):
    """
    Scatter pillar features into dense (B, C, H, W) BEV grid.
    Fully differentiable (no in-place index_put_ on leaf tensors).
    Deduplicates coords before scatter for DDP determinism.
    """
    def __init__(self, bev_h: int, bev_w: int):
        super().__init__()
        self.H = bev_h
        self.W = bev_w

    def forward(self, pillar_feats: torch.Tensor,
                coords: torch.Tensor,
                batch_size: int) -> torch.Tensor:
        C      = pillar_feats.shape[1]
        dtype  = pillar_feats.dtype
        device = pillar_feats.device

        b_idx  = coords[:, 0].long()
        r_idx  = coords[:, 1].long().clamp(0, self.H - 1)
        c_idx  = coords[:, 2].long().clamp(0, self.W - 1)

        # Deduplicate coords
        flat_key    = b_idx * (self.H * self.W) + r_idx * self.W + c_idx
        _, inv_idx  = torch.unique(flat_key.flip(0), return_inverse=True)
        n_unique    = inv_idx.max().item() + 1
        keep_rev    = torch.zeros(n_unique, dtype=torch.long, device=device)
        keep_rev.scatter_(0, inv_idx,
                          torch.arange(len(inv_idx), device=device))
        keep_mask_rev = torch.zeros(len(flat_key), dtype=torch.bool, device=device)
        keep_mask_rev[keep_rev] = True
        keep_mask   = keep_mask_rev.flip(0)

        pillar_feats = pillar_feats[keep_mask]
        b_idx        = b_idx[keep_mask]
        r_idx        = r_idx[keep_mask]
        c_idx        = c_idx[keep_mask]

        flat_idx     = b_idx * (self.H * self.W) + r_idx * self.W + c_idx
        flat_idx_exp = flat_idx.unsqueeze(1).expand(-1, C)
        bev_flat     = torch.zeros(batch_size * self.H * self.W, C,
                                   dtype=dtype, device=device)
        bev_flat     = bev_flat.scatter(0, flat_idx_exp, pillar_feats)
        bev = bev_flat.view(batch_size, self.H, self.W, C).permute(0, 3, 1, 2).contiguous()
        return bev


# ===========================================================================
# 3.  Wave Propagation Operator (WPO)
# ===========================================================================

def _stable_sinc(omega_d: torch.Tensor, t: float) -> torch.Tensor:
    """
    Numerically stable sin(ωd·t)/ωd via Taylor expansion for small ωd.
    Prevents Inf/NaN in forward and backward.
    """
    x      = omega_d * t
    x_safe = torch.where(x < _SINC_TAYLOR_THRESH,
                         x, torch.zeros_like(x))
    x2     = x_safe * x_safe
    taylor = 1.0 - x2 / 6.0 + (x2 * x2) / 120.0
    exact  = torch.sin(x) / x.clamp(min=_SINC_TAYLOR_THRESH)
    return torch.where(x < _SINC_TAYLOR_THRESH, taylor, exact)


class WavePropagationOperator(nn.Module):
    """
    Implements the underdamped wave equation in Fourier domain (Eq.6, WaveFormer):

        U^t = F⁻¹{ e^{-α/2·t} [ F(U⁰)·cos(ωd·t)
                    + sin(ωd·t)/ωd · (F(V⁰) + α/2·F(U⁰)) ] }

    where ωd = sqrt(v²·(ωx²+ωy²) − (α/2)²)

    Key stability fixes:
        - _stable_sinc for ωd → 0
        - Explicit float32 enforcement in spectral path
        - Spectral magnitude clamping before irfft2 (bounds backward grad)
        - Post-irfft2 output clamp
        - GroupNorm instead of BatchNorm (no batch-size sensitivity)
    """
    def __init__(self, channels: int, bev_h: int, bev_w: int,
                 t: float = 1.0,
                 init_alpha: float = 0.1,
                 init_v: float = 1.0):
        super().__init__()
        self.C   = channels
        self.H   = bev_h
        self.W   = bev_w
        self.t   = t

        self.log_alpha = nn.Parameter(torch.tensor(math.log(init_alpha)))
        self.log_v     = nn.Parameter(torch.tensor(math.log(init_v)))

        self.dw_conv  = nn.Conv2d(channels, channels, kernel_size=3,
                                   padding=1, groups=channels, bias=False)
        self.v0_proj  = nn.Sequential(
            nn.Conv2d(channels, channels, 1, bias=False),
            nn.GroupNorm(min(8, channels), channels),
        )
        self.out_proj = nn.Sequential(
            nn.Conv2d(channels, channels, 1, bias=False),
            nn.GroupNorm(min(8, channels), channels),
            nn.SiLU(inplace=True),
        )

        freq_y = torch.fft.fftfreq(bev_h)
        freq_x = torch.fft.rfftfreq(bev_w)
        wy, wx = torch.meshgrid(freq_y, freq_x, indexing='ij')
        omega2 = (wx ** 2 + wy ** 2).unsqueeze(0).unsqueeze(0)
        self.register_buffer('omega2', omega2)

    def _wave_propagate(self, U0_f: torch.Tensor,
                        V0_f: torch.Tensor,
                        alpha: torch.Tensor,
                        v: torch.Tensor) -> torch.Tensor:
        assert U0_f.dtype == torch.complex64
        assert V0_f.dtype == torch.complex64

        alpha  = alpha.float()
        v      = v.float()
        t      = self.t
        half_a = alpha / 2.0

        omega2_d = torch.clamp((v ** 2) * self.omega2 - half_a ** 2, min=EPS)
        omega_d  = torch.sqrt(omega2_d)

        decay    = torch.exp(-half_a * t)
        cos_term = torch.cos(omega_d * t)
        sinc_val = _stable_sinc(omega_d, t)

        Ut_f = decay * (U0_f * cos_term + sinc_val * (V0_f + half_a * U0_f))

        # Magnitude clamp — preserves phase, bounds irfft2 backward gradient
        mag   = Ut_f.abs()
        scale = _SPECTRAL_CLAMP / torch.clamp(mag, min=_SPECTRAL_CLAMP)
        Ut_f  = Ut_f * scale

        return Ut_f

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        dtype = x.dtype

        U0 = self.dw_conv(x)
        V0 = self.v0_proj(x)

        with torch.amp.autocast('cuda', enabled=False):
            U0_f32 = U0.float()
            V0_f32 = V0.float()

            U0_f = torch.fft.rfft2(U0_f32, norm='ortho')
            V0_f = torch.fft.rfft2(V0_f32, norm='ortho')

            alpha = torch.exp(self.log_alpha)
            v     = torch.exp(self.log_v)

            Ut_f   = self._wave_propagate(U0_f, V0_f, alpha, v)
            Ut_f32 = torch.fft.irfft2(Ut_f, s=(self.H, self.W), norm='ortho')
            Ut_f32 = torch.clamp(Ut_f32, min=-256.0, max=256.0)

        Ut = Ut_f32.to(dtype)

        out_p = self.out_proj(Ut)
        out_p = torch.clamp(out_p, min=-65000.0, max=65000.0)

        return out_p + x


# ===========================================================================
# 4.  WPO Block  (WPO + LayerNorm + FFN, with gradient checkpointing)
# ===========================================================================
class WPOBlock(nn.Module):
    def __init__(self, channels: int, bev_h: int, bev_w: int,
                 expansion: int = 4, use_checkpoint: bool = True):
        super().__init__()
        self.use_checkpoint = use_checkpoint

        self.norm1 = nn.LayerNorm(channels)
        self.wpo   = WavePropagationOperator(channels, bev_h, bev_w)
        self.norm2 = nn.LayerNorm(channels)
        self.ffn   = nn.Sequential(
            nn.Linear(channels, channels * expansion),
            nn.SiLU(inplace=True),
            nn.Linear(channels * expansion, channels),
        )

    def _inner_forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x_ln = self.norm1(x.permute(0, 2, 3, 1)).permute(0, 3, 1, 2)
        x    = residual + self.wpo(x_ln)

        residual = x
        x_ln = self.norm2(x.permute(0, 2, 3, 1))
        x_ff = self.ffn(x_ln)
        x    = residual + x_ff.permute(0, 3, 1, 2)
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        should_checkpoint = (
            self.use_checkpoint
            and self.training
            and (x.requires_grad or torch.is_grad_enabled())
        )
        if should_checkpoint:
            return checkpoint.checkpoint(self._inner_forward, x,
                                         use_reentrant=False)
        return self._inner_forward(x)


# ===========================================================================
# 5.  WPO Backbone  (4-stage hierarchical)
# ===========================================================================
class WPOBackbone(nn.Module):
    """
    4-stage hierarchical backbone.
    Stage i: [WPOBlock × depth[i]] → PatchMerge (2× spatial downsample)
    All norms are GroupNorm — no batch-size sensitivity.
    """
    def __init__(self,
                 in_channels:   int       = 64,
                 stage_dims:    list      = [96, 192, 384, 768],
                 depths:        list      = [2, 2, 6, 2],
                 bev_h:         int       = 512,
                 bev_w:         int       = 512,
                 use_checkpoint: bool     = True):
        super().__init__()

        self.stages      = nn.ModuleList()
        self.downsamples = nn.ModuleList()
        self.stem        = nn.Sequential(
            nn.Conv2d(in_channels, stage_dims[0], 3, padding=1, bias=False),
            nn.GroupNorm(min(8, stage_dims[0]), stage_dims[0]),
            nn.SiLU(inplace=True),
        )

        h, w = bev_h, bev_w
        for i, (dim, depth) in enumerate(zip(stage_dims, depths)):
            stage = nn.Sequential(*[
                WPOBlock(dim, h, w, use_checkpoint=use_checkpoint)
                for _ in range(depth)
            ])
            self.stages.append(stage)
            if i < len(stage_dims) - 1:
                next_dim = stage_dims[i + 1]
                self.downsamples.append(nn.Sequential(
                    nn.Conv2d(dim, next_dim, kernel_size=2, stride=2, bias=False),
                    nn.GroupNorm(min(8, next_dim), next_dim),
                ))
                h, w = h // 2, w // 2
            else:
                self.downsamples.append(nn.Identity())

        self.out_dims = stage_dims

    def forward(self, x: torch.Tensor) -> list:
        x = self.stem(x)
        outs = []
        for stage, down in zip(self.stages, self.downsamples):
            x = stage(x)
            outs.append(x)
            x = down(x)
        return outs


# ===========================================================================
# 6.  CenterPoint-style Detection Head
# ===========================================================================
class CenterPointHead(nn.Module):
    """
    Simplified CenterPoint head: heatmap (3 classes) + 8-D regression.
    GroupNorm for batch-size robustness.
    """
    NUM_CLASSES = 3
    REG_DIMS    = 8

    def __init__(self, in_channels: int, hidden: int = 256):
        super().__init__()
        self.shared = nn.Sequential(
            nn.Conv2d(in_channels, hidden, 3, padding=1, bias=False),
            nn.GroupNorm(min(8, hidden), hidden),
            nn.SiLU(inplace=True),
        )
        self.heatmap = nn.Conv2d(hidden, self.NUM_CLASSES, 1)
        self.reg     = nn.Conv2d(hidden, self.REG_DIMS,    1)

    def forward(self, feat: torch.Tensor) -> dict:
        h = self.shared(feat)
        return {
            'heatmap': self.heatmap(h),
            'reg'    : self.reg(h),
        }


# ===========================================================================
# 7.  BEVWaveFormer — Full End-to-End Model
# ===========================================================================
class BEVWaveFormer(nn.Module):
    """
    Complete BEV-WaveFormer model.

    Pipeline:
        pillars → PillarFeatureNet → BEVScatter → WPOBackbone → FPN → CenterPointHead
    """
    def __init__(self,
                 pillar_in_ch:   int   = 4,
                 pillar_out_ch:  int   = 64,
                 max_points:     int   = 32,
                 bev_h:          int   = 512,
                 bev_w:          int   = 512,
                 stage_dims:     list  = [96, 192, 384, 768],
                 depths:         list  = [2, 2, 6, 2],
                 use_checkpoint: bool  = True):
        super().__init__()

        self.vfe      = PillarFeatureNet(pillar_in_ch, pillar_out_ch, max_points)
        self.scatter  = BEVScatter(bev_h, bev_w)
        self.backbone = WPOBackbone(pillar_out_ch, stage_dims, depths,
                                    bev_h, bev_w, use_checkpoint)

        # Lightweight U-Net FPN Neck (64ch to save VRAM)
        fpn_dim = 64

        self.fpn_s4_shrink = nn.Sequential(
            nn.Conv2d(stage_dims[3], fpn_dim, 1, bias=False),
            nn.GroupNorm(8, fpn_dim), nn.SiLU(inplace=True))
        self.fpn_s4_up = nn.Sequential(
            nn.ConvTranspose2d(fpn_dim, fpn_dim, kernel_size=2, stride=2),
            nn.GroupNorm(8, fpn_dim), nn.SiLU(inplace=True))

        self.fpn_s3_shrink = nn.Sequential(
            nn.Conv2d(stage_dims[2], fpn_dim, 1, bias=False),
            nn.GroupNorm(8, fpn_dim), nn.SiLU(inplace=True))
        self.fpn_s3_up = nn.Sequential(
            nn.ConvTranspose2d(fpn_dim, fpn_dim, kernel_size=2, stride=2),
            nn.GroupNorm(8, fpn_dim), nn.SiLU(inplace=True))

        self.fpn_s2_shrink = nn.Sequential(
            nn.Conv2d(stage_dims[1], fpn_dim, 1, bias=False),
            nn.GroupNorm(8, fpn_dim), nn.SiLU(inplace=True))

        self.head = CenterPointHead(in_channels=fpn_dim)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
                m.eps = 1e-3
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, (nn.GroupNorm, nn.LayerNorm)):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
        nn.init.constant_(self.head.heatmap.bias, -2.19)

    def forward(self, pillars: torch.Tensor,
                num_points: torch.Tensor,
                coords: torch.Tensor,
                batch_size: int) -> dict:

        pillar_feats      = self.vfe(pillars, num_points)
        bev               = self.scatter(pillar_feats, coords, batch_size)
        multi_scale_feats = self.backbone(bev)

        s2 = multi_scale_feats[1]   # (B, 192, 256, 256)
        s3 = multi_scale_feats[2]   # (B, 384, 128, 128)
        s4 = multi_scale_feats[3]   # (B, 768,  64,  64)

        f4        = self.fpn_s4_up(self.fpn_s4_shrink(s4))         # 64→128
        f3        = self.fpn_s3_up(f4 + self.fpn_s3_shrink(s3))    # 128→256
        fused_256 = f3 + self.fpn_s2_shrink(s2)

        return self.head(fused_256)


print(" model.py cell loaded.")

 model.py cell loaded.


In [6]:
# ============================================================
# CELL 5 — train.py  (loss, training loop, checkpointing)
# ============================================================
import os
import io
import logging
import math
import time
import signal
from pathlib import Path

import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.amp import autocast


# ===========================================================================
# Logging — only rank 0 writes
# ===========================================================================
def get_logger(rank: int, log_file: str = 'train.log') -> logging.Logger:
    logger = logging.getLogger('BEVWaveFormer')
    logger.setLevel(logging.INFO)
    # Remove any existing handlers to avoid duplication across notebook re-runs
    logger.handlers.clear()
    if rank == 0:
        fmt = logging.Formatter('[%(asctime)s][%(levelname)s] %(message)s',
                                datefmt='%Y-%m-%d %H:%M:%S')
        ch = logging.StreamHandler()
        ch.setFormatter(fmt)
        logger.addHandler(ch)
        fh = logging.FileHandler(log_file)
        fh.setFormatter(fmt)
        logger.addHandler(fh)
    else:
        logger.addHandler(logging.NullHandler())
    return logger


# ===========================================================================
# Gaussian heatmap rendering
# ===========================================================================
def gaussian_radius(det_size: tuple, min_overlap: float = 0.7) -> float:
    h, w = det_size
    a1  = 1.0;  b1 = h + w;  c1 = h * w * (1 - min_overlap) / (1 + min_overlap)
    sq1 = math.sqrt(max(b1 ** 2 - 4 * a1 * c1, 0.0));  r1 = (b1 - sq1) / (2 * a1)
    a2  = 4.0;  b2 = 2 * (h + w);  c2 = (1 - min_overlap) * h * w
    sq2 = math.sqrt(max(b2 ** 2 - 4 * a2 * c2, 0.0));  r2 = (b2 - sq2) / (2 * a2)
    a3  = 4 * min_overlap;  b3 = -2 * min_overlap * (h + w);  c3 = (min_overlap - 1) * h * w
    sq3 = math.sqrt(max(b3 ** 2 - 4 * a3 * c3, 0.0));  r3 = (b3 + sq3) / (2 * a3)
    return max(0.0, min(r1, r2, r3))


def draw_gaussian(heatmap: torch.Tensor, center_x: int, center_y: int,
                  radius: int) -> None:
    H, W   = heatmap.shape
    sigma  = max(radius / 3.0, 1e-2)
    diameter = 2 * radius + 1
    x      = torch.arange(diameter, dtype=torch.float32) - radius
    g1d    = torch.exp(-x ** 2 / (2 * sigma ** 2))
    g2d    = g1d[:, None] * g1d[None, :]

    x0, y0 = center_x - radius, center_y - radius
    x1, y1 = center_x + radius + 1, center_y + radius + 1
    hm_x0 = max(x0, 0);  hm_x1 = min(x1, W)
    hm_y0 = max(y0, 0);  hm_y1 = min(y1, H)
    if hm_x0 >= hm_x1 or hm_y0 >= hm_y1:
        return

    k_x0 = hm_x0 - x0;  k_x1 = k_x0 + (hm_x1 - hm_x0)
    k_y0 = hm_y0 - y0;  k_y1 = k_y0 + (hm_y1 - hm_y0)

    kernel_patch = g2d[k_y0:k_y1, k_x0:k_x1].to(heatmap.device)
    heatmap[hm_y0:hm_y1, hm_x0:hm_x1] = torch.maximum(
        heatmap[hm_y0:hm_y1, hm_x0:hm_x1], kernel_patch)


def build_gt_targets(batch_boxes, batch_size, num_classes,
                     hm_h, hm_w, reg_dims, bev_cfg, device) -> dict:
    """Build Gaussian heatmap + regression targets from GT boxes (all float32)."""
    heatmap  = torch.zeros(batch_size, num_classes, hm_h, hm_w,
                           dtype=torch.float32, device=device)
    reg      = torch.zeros(batch_size, reg_dims,    hm_h, hm_w,
                           dtype=torch.float32, device=device)
    reg_mask = torch.zeros(batch_size, 1,           hm_h, hm_w,
                           dtype=torch.float32, device=device)

    x_min, x_max = bev_cfg['x_min'], bev_cfg['x_max']
    y_min, y_max = bev_cfg['y_min'], bev_cfg['y_max']
    ps_x         = bev_cfg['pillar_size_x']
    ps_y         = bev_cfg['pillar_size_y']
    scale_x = hm_w / bev_cfg['bev_w']
    scale_y = hm_h / bev_cfg['bev_h']
    cell_x  = ps_x / scale_x
    cell_y  = ps_y / scale_y

    for b_idx, boxes in enumerate(batch_boxes):
        if boxes is None or len(boxes) == 0:
            continue
        boxes_np = boxes.cpu() if isinstance(boxes, torch.Tensor) else boxes
        for box in boxes_np:
            cls_id = int(box[0])
            bev_x  = float(box[1]);  bev_y  = float(box[2])
            lz     = float(box[3]);  box_w  = float(box[4])
            box_l  = float(box[5]);  box_h  = float(box[6])
            yaw    = float(box[7])

            if not (x_min <= bev_x < x_max and y_min <= bev_y < y_max):
                continue

            exact_px = int((bev_x - x_min) / ps_x * (hm_w / bev_cfg['bev_w']))
            exact_py = int((bev_y - y_min) / ps_y * (hm_h / bev_cfg['bev_h']))
            px = max(0, min(exact_px, hm_w - 1))
            py = max(0, min(exact_py, hm_h - 1))

            r_x    = max(1, int(box_l / (2 * cell_x)))
            r_y    = max(1, int(box_w / (2 * cell_y)))
            radius = max(0, int(gaussian_radius((r_y * 2, r_x * 2))))

            if 0 <= cls_id < num_classes:
                draw_gaussian(heatmap[b_idx, cls_id], px, py, radius)

            reg[b_idx, 0, py, px] = exact_px - px
            reg[b_idx, 1, py, px] = exact_py - py
            reg[b_idx, 2, py, px] = lz
            reg[b_idx, 3, py, px] = math.log(max(box_w, 1e-3))
            reg[b_idx, 4, py, px] = math.log(max(box_l, 1e-3))
            reg[b_idx, 5, py, px] = math.log(max(box_h, 1e-3))
            reg[b_idx, 6, py, px] = math.sin(yaw)
            reg[b_idx, 7, py, px] = math.cos(yaw)
            reg_mask[b_idx, 0, py, px] = 1.0

    return {'heatmap': heatmap, 'reg': reg, 'reg_mask': reg_mask}


# ===========================================================================
# Loss
# ===========================================================================
class BEVLoss(nn.Module):
    """
    Focal loss (heatmap) + SmoothL1 (regression).
    Both pred and target explicitly cast to float32 before loss computation
    to prevent dtype mismatch overflow under bfloat16 autocast.
    """
    def __init__(self, alpha: float = 2.0, beta: float = 4.0,
                 reg_weight: float = 2.0):
        super().__init__()
        self.alpha      = alpha
        self.beta       = beta
        self.reg_weight = reg_weight

    def focal_loss(self, pred: torch.Tensor,
                   target: torch.Tensor) -> torch.Tensor:
        pred   = torch.clamp(pred.float().sigmoid(), min=1e-4, max=1 - 1e-4)
        target = target.float()
        pos_mask = (target == 1).float()
        neg_mask = (target  < 1).float()
        pos_loss = torch.log(pred)     * (1 - pred) ** self.alpha * pos_mask
        neg_loss = torch.log(1 - pred) * pred ** self.alpha \
                   * (1 - target) ** self.beta * neg_mask
        num_pos = pos_mask.sum().clamp(min=1)
        return -(pos_loss.sum() + neg_loss.sum()) / num_pos

    def forward(self, preds: dict, targets: dict) -> torch.Tensor:
        hm_loss  = self.focal_loss(preds['heatmap'], targets['heatmap'])
        mask     = targets['reg_mask'].float()
        num_pos  = mask.sum().clamp(min=1)
        reg_pred = preds['reg'].float() * mask
        reg_tgt  = targets['reg'].float() * mask
        reg_loss = nn.functional.smooth_l1_loss(
            reg_pred, reg_tgt, reduction='sum') / num_pos
        return hm_loss + self.reg_weight * reg_loss


# ===========================================================================
# Checkpointing
# ===========================================================================
def save_checkpoint(model, optimizer, epoch, loss, save_dir, rank,
                    filename=None):
    """Atomic checkpoint save on rank 0 only."""
    if rank != 0:
        return
    os.makedirs(save_dir, exist_ok=True)
    if filename is None:
        filename = f'ckpt_epoch{epoch:04d}.pth'
    path = os.path.join(save_dir, filename)
    state = {
        'epoch'           : epoch,
        'model_state'     : model.module.state_dict(),
        'optimizer_state' : optimizer.state_dict(),
        'loss'            : loss,
    }
    tmp_path = path + '.tmp'
    torch.save(state, tmp_path)
    os.replace(tmp_path, path)

    latest_path = os.path.join(save_dir, 'latest.pth')
    if os.path.islink(latest_path) or os.path.exists(latest_path):
        os.remove(latest_path)
    os.symlink(os.path.abspath(path), latest_path)


def load_checkpoint(model, optimizer, checkpoint_path, device, logger) -> int:
    """Load checkpoint, return next epoch. CPU-first to avoid VRAM spike."""
    logger.info(f'Loading checkpoint: {checkpoint_path}')
    ckpt = torch.load(checkpoint_path, map_location='cpu')
    model.load_state_dict(ckpt['model_state'])
    model.to(device)
    optimizer.load_state_dict(ckpt['optimizer_state'])
    start_epoch = ckpt['epoch'] + 1
    logger.info(f'Resumed from epoch {ckpt["epoch"]}, loss={ckpt["loss"]:.4f}')
    return start_epoch


# ===========================================================================
# Session time-limit save  (11-hour Kaggle guard)
# ===========================================================================
class SessionTimer:
    """
    Tracks wall-clock time since training started.
    Call .should_save_and_exit() each epoch to detect the 11-hour limit.
    When triggered, saves a 'session_checkpoint.pth' and signals the loop
    to stop cleanly so the next Kaggle session can resume from it.
    """
    def __init__(self, limit_hours: float = 11.0):
        self.limit_seconds = limit_hours * 3600
        self.start_time    = time.time()

    def elapsed(self) -> float:
        return time.time() - self.start_time

    def should_save_and_exit(self) -> bool:
        return self.elapsed() >= self.limit_seconds

    def remaining_str(self) -> str:
        rem = max(0.0, self.limit_seconds - self.elapsed())
        h, r = divmod(int(rem), 3600)
        m, s = divmod(r, 60)
        return f'{h:02d}h{m:02d}m{s:02d}s'


def save_session_checkpoint(model, optimizer, scheduler, epoch,
                             val_loss, save_dir, rank, logger):
    """
    Save everything needed to resume in the next Kaggle session.
    Saves on rank 0 only.  Also copies the training CSV.
    """
    if rank != 0:
        return
    os.makedirs(save_dir, exist_ok=True)
    path = os.path.join(save_dir, 'session_checkpoint.pth')
    state = {
        'epoch'           : epoch,
        'model_state'     : model.module.state_dict(),
        'optimizer_state' : optimizer.state_dict(),
        'scheduler_state' : scheduler.state_dict(),
        'loss'            : val_loss,
    }
    tmp = path + '.tmp'
    torch.save(state, tmp)
    os.replace(tmp, path)
    logger.info(f' Session checkpoint saved → {path}  (epoch {epoch})')

    # Also copy the metrics CSV so it survives the session
    csv_src = os.path.join(save_dir, 'train_metrics.csv')
    csv_dst = os.path.join(save_dir, 'train_metrics_session_backup.csv')
    if os.path.exists(csv_src):
        import shutil
        shutil.copy2(csv_src, csv_dst)
        logger.info(f' Metrics CSV backed up → {csv_dst}')


# ===========================================================================
# Training loop (one epoch)
# ===========================================================================
def train_one_epoch(model, loader, optimizer, criterion,
                    device, epoch, logger, log_interval=50):
    """
    One full training epoch.

    Option D — bfloat16 autocast, no GradScaler.
    RTX Pro 6000 (Ada Lovelace) supports BF16 Tensor Cores natively.
    BF16 has the same exponent range as FP32 → no overflow/underflow issues
    that plagued FP16 GradScaler on sparse BEV grids.

    NaN handling:
        Detect via grad_norm and param finiteness.
        On NaN: roll back model + optimizer to last CPU-RAM safe state.
        The safe state is a BytesIO buffer updated every 50 clean steps.

    grad_norm: fixed at 7.0 (relaxed from the original warm-up schedule
    because BF16 eliminates the gradient spike risk that motivated the warm-up).
    """
    model.train()
    total_loss = 0.0
    skipped    = 0
    t0         = time.time()
    max_norm   = 7.0

    # ── Rolling safe state in CPU RAM (uses Kaggle's 175GB system RAM) ──────
    safe_state_buffer = io.BytesIO()
    torch.save({
        'model'    : model.module.state_dict(),
        'optimizer': optimizer.state_dict()
    }, safe_state_buffer)

    for step, batch in enumerate(loader):
        pillars    = batch['pillars'].to(device,    non_blocking=True)
        num_points = batch['num_points'].to(device, non_blocking=True)
        coords     = batch['coords'].to(device,     non_blocking=True)
        bs         = batch['batch_size']

        optimizer.zero_grad(set_to_none=True)

        # ── Forward under bfloat16 AMP ────────────────────────────────────
        with autocast("cuda", dtype=torch.bfloat16):
            preds = model(pillars, num_points, coords, bs)
            H_out = preds['heatmap'].shape[-2]
            W_out = preds['heatmap'].shape[-1]
            targets = build_gt_targets(
                batch['boxes'], bs, 3, H_out, W_out, 8, BEV_CFG, device)
            loss = criterion(preds, targets)

        # ── Backward ─────────────────────────────────────────────────────
        loss.backward()

        # ── Gradient check + clip ─────────────────────────────────────────
        grad_norm = torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=max_norm)

        # CHECK 1: NaN gradient?
        # CHECK 2: Did the previous step push any weight to NaN?
        has_nan_param = any(
            not torch.isfinite(p).all() for p in model.parameters())

        if not torch.isfinite(grad_norm) or has_nan_param:
            optimizer.zero_grad(set_to_none=True)
            logger.warning(
                f'Epoch {epoch} Step {step}: NaN detected. '
                f'Rolling back model+optimizer to last safe state.')

            safe_state_buffer.seek(0)
            ckpt_safe = torch.load(safe_state_buffer, map_location=device)
            model.module.load_state_dict(ckpt_safe['model'])
            optimizer.load_state_dict(ckpt_safe['optimizer'])
            for p in model.parameters():
                p.grad = None
            skipped += 1
            continue

        # ── DDP-safe spectral param freeze (epochs 0–4) ───────────────────
        if epoch < 5:
            for name, param in model.named_parameters():
                if ('log_alpha' in name or 'log_v' in name) and param.grad is not None:
                    param.grad.zero_()

        optimizer.step()

        # ── Post-step physics clamp on spectral params ────────────────────
        with torch.no_grad():
            for name, param in model.named_parameters():
                if 'log_alpha' in name or 'log_v' in name:
                    param.clamp_(-4.0, 3.0)

        # ── Update rolling safe state every 50 clean steps ────────────────
        if step > 0 and step % 50 == 0:
            if (torch.isfinite(grad_norm)
                    and torch.isfinite(torch.tensor(loss.item()))):
                safe_state_buffer.seek(0)
                safe_state_buffer.truncate()
                torch.save({
                    'model'    : model.module.state_dict(),
                    'optimizer': optimizer.state_dict()
                }, safe_state_buffer)
            else:
                logger.info(
                    f'Step {step}: Skipped safe_state update '
                    f'(grad_norm={grad_norm:.1f}, loss={loss.item():.2f})')

        total_loss += loss.item()

        if step % log_interval == 0:
            elapsed = time.time() - t0
            logger.info(
                f'Epoch {epoch:04d} | Step {step:05d}/{len(loader):05d} '
                f'| loss={loss.item():.4f} '
                f'| grad_norm={grad_norm:.3f} '
                f'| max_norm={max_norm:.3f} '
                f'| {elapsed:.1f}s elapsed')

            if dist.get_rank() == 0:
                csv_path = os.path.join(SAVE_DIR, 'train_metrics.csv')
                if not os.path.exists(csv_path):
                    with open(csv_path, 'w') as f:
                        f.write("Epoch,Step,TrainLoss,GradNorm,MaxNorm\n")
                with open(csv_path, 'a') as f:
                    f.write(f"{epoch},{step},{loss.item():.4f},"
                            f"{grad_norm:.3f},{max_norm:.3f}\n")

    if skipped > 0:
        logger.warning(
            f'Epoch {epoch:04d} | Skipped {skipped} steps '
            f'({100*skipped/max(len(loader),1):.1f}%) due to non-finite gradients.')

    return total_loss / max(len(loader) - skipped, 1)


# ===========================================================================
# Validation loop
# ===========================================================================
@torch.no_grad()
def validate(model, loader, criterion, device, epoch, logger) -> float:
    """Validation with all-reduce averaging across DDP ranks."""
    model.eval()
    total_loss = torch.tensor(0.0, device=device)
    count      = torch.tensor(0,   device=device)

    for batch in loader:
        pillars    = batch['pillars'].to(device,    non_blocking=True)
        num_points = batch['num_points'].to(device, non_blocking=True)
        coords     = batch['coords'].to(device,     non_blocking=True)
        bs         = batch['batch_size']

        with autocast("cuda", dtype=torch.bfloat16):
            preds = model(pillars, num_points, coords, bs)
            H_out = preds['heatmap'].shape[-2]
            W_out = preds['heatmap'].shape[-1]
            targets = build_gt_targets(
                batch['boxes'], bs, 3, H_out, W_out, 8, BEV_CFG, device)
            loss = criterion(preds, targets)

        total_loss += loss
        count      += 1

    dist.all_reduce(total_loss, op=dist.ReduceOp.SUM)
    dist.all_reduce(count,      op=dist.ReduceOp.SUM)
    avg_loss = (total_loss / count).item()
    logger.info(f'Epoch {epoch:04d} | VAL loss={avg_loss:.4f}')
    return avg_loss


print("train.py cell loaded.")

train.py cell loaded.


In [7]:
# ============================================================
# CELL 6 — validate_checkpoint.py  (standalone DDP validation)
# ============================================================
import math
import torch.nn.functional as F

CLASS_NAMES = ['Car', 'Pedestrian', 'Cyclist']


def focal_loss_val(pred, target, alpha=2.0, beta=4.0):
    pred     = torch.clamp(pred.float().sigmoid(), min=1e-4, max=1 - 1e-4)
    target   = target.float()
    pos_mask = (target == 1).float()
    neg_mask = (target  < 1).float()
    pos_loss = torch.log(pred)     * (1 - pred) ** alpha * pos_mask
    neg_loss = torch.log(1 - pred) * pred ** alpha \
               * (1 - target) ** beta * neg_mask
    num_pos  = pos_mask.sum().clamp(min=1)
    return -(pos_loss.sum() + neg_loss.sum()) / num_pos


def _gaussian_radius_val(h, w, min_overlap=0.7):
    a1=1.0; b1=h+w; c1=h*w*(1-min_overlap)/(1+min_overlap)
    sq1=math.sqrt(max(b1**2-4*a1*c1,0)); r1=(b1-sq1)/(2*a1)
    a2=4.0; b2=2*(h+w); c2=(1-min_overlap)*h*w
    sq2=math.sqrt(max(b2**2-4*a2*c2,0)); r2=(b2-sq2)/(2*a2)
    a3=4*min_overlap; b3=-2*min_overlap*(h+w); c3=(min_overlap-1)*h*w
    sq3=math.sqrt(max(b3**2-4*a3*c3,0)); r3=(b3+sq3)/(2*a3)
    return max(0, int(min(r1,r2,r3)))


def _draw_gaussian_val(heatmap, cx, cy, radius):
    H,W=heatmap.shape; sigma=max(radius/3.0,1e-2); d=2*radius+1
    x=torch.arange(d,dtype=torch.float32)-radius
    g2d=(torch.exp(-x**2/(2*sigma**2))[:,None])*(torch.exp(-x**2/(2*sigma**2))[None,:])
    x0,y0=cx-radius,cy-radius; x1,y1=cx+radius+1,cy+radius+1
    hx0,hx1=max(x0,0),min(x1,W); hy0,hy1=max(y0,0),min(y1,H)
    if hx0>=hx1 or hy0>=hy1: return
    patch=g2d[hy0-y0:hy0-y0+(hy1-hy0), hx0-x0:hx0-x0+(hx1-hx0)].to(heatmap.device)
    heatmap[hy0:hy1,hx0:hx1]=torch.maximum(heatmap[hy0:hy1,hx0:hx1],patch)


def build_targets_val(batch_boxes, batch_size, num_classes, hm_h, hm_w,
                      bev_cfg, device):
    heatmap  = torch.zeros(batch_size, num_classes, hm_h, hm_w, device=device)
    reg      = torch.zeros(batch_size, 8,           hm_h, hm_w, device=device)
    reg_mask = torch.zeros(batch_size, 1,           hm_h, hm_w, device=device)
    x_min,x_max = bev_cfg['x_min'],bev_cfg['x_max']
    y_min,y_max = bev_cfg['y_min'],bev_cfg['y_max']
    ps_x,ps_y   = bev_cfg['pillar_size_x'],bev_cfg['pillar_size_y']
    cell_x = ps_x/(hm_w/bev_cfg['bev_w'])
    cell_y = ps_y/(hm_h/bev_cfg['bev_h'])
    for b,boxes in enumerate(batch_boxes):
        if boxes is None or len(boxes)==0: continue
        for box in (boxes.cpu() if isinstance(boxes,torch.Tensor) else boxes):
            cls_id=int(box[0]); bev_x,bev_y=float(box[1]),float(box[2])
            lz=float(box[3]); bw,bl,bh,yaw=float(box[4]),float(box[5]),float(box[6]),float(box[7])
            if not (x_min<=bev_x<x_max and y_min<=bev_y<y_max): continue
            exact_px=(bev_x-x_min)/ps_x*(hm_w/bev_cfg['bev_w'])
            exact_py=(bev_y-y_min)/ps_y*(hm_h/bev_cfg['bev_h'])
            px=max(0,min(int(exact_px),hm_w-1)); py=max(0,min(int(exact_py),hm_h-1))
            r=_gaussian_radius_val(max(1,int(bw/(2*cell_y)))*2, max(1,int(bl/(2*cell_x)))*2)
            if 0<=cls_id<num_classes: _draw_gaussian_val(heatmap[b,cls_id],px,py,r)
            reg[b,0,py,px]=exact_px-px; reg[b,1,py,px]=exact_py-py; reg[b,2,py,px]=lz
            reg[b,3,py,px]=math.log(max(bw,1e-3)); reg[b,4,py,px]=math.log(max(bl,1e-3))
            reg[b,5,py,px]=math.log(max(bh,1e-3)); reg[b,6,py,px]=math.sin(yaw)
            reg[b,7,py,px]=math.cos(yaw); reg_mask[b,0,py,px]=1.0
    return heatmap, reg, reg_mask


class ValMetrics:
    """Accumulates per-rank stats; aggregate() all-reduces across ranks."""
    def __init__(self, num_classes: int):
        self.C = num_classes
        self.focal_sum   = 0.0;  self.reg_sum = 0.0;  self.n_batches = 0
        self.gt_response = [0.0] * num_classes
        self.gt_count    = [0]   * num_classes
        self.recall_03   = [0]   * num_classes
        self.recall_05   = [0]   * num_classes

    @torch.no_grad()
    def update(self, pred_hm, gt_hm, pred_reg, gt_reg, reg_mask):
        self.focal_sum += focal_loss_val(pred_hm, gt_hm).item()
        mask    = reg_mask.float()
        num_pos = mask.sum().clamp(min=1)
        self.reg_sum += (F.smooth_l1_loss(
            pred_reg.float()*mask, gt_reg.float()*mask,
            reduction='sum') / num_pos).item()
        self.n_batches += 1
        pred_prob = pred_hm.float().sigmoid()
        for c in range(self.C):
            centres = (gt_hm[:, c] == 1.0)
            if centres.any():
                resp = pred_prob[:, c][centres]
                self.gt_response[c] += resp.sum().item()
                self.gt_count[c]    += centres.sum().item()
                self.recall_03[c]   += (resp >= 0.3).sum().item()
                self.recall_05[c]   += (resp >= 0.5).sum().item()

    def aggregate(self, device):
        def _r(val):
            t = torch.tensor(float(val), device=device)
            dist.all_reduce(t, op=dist.ReduceOp.SUM)
            return t.item()
        self.focal_sum  = _r(self.focal_sum)
        self.reg_sum    = _r(self.reg_sum)
        self.n_batches  = int(_r(self.n_batches))
        for c in range(self.C):
            self.gt_response[c] = _r(self.gt_response[c])
            self.gt_count[c]    = int(_r(self.gt_count[c]))
            self.recall_03[c]   = int(_r(self.recall_03[c]))
            self.recall_05[c]   = int(_r(self.recall_05[c]))

    def report(self, logger, epoch=None, save_dir=None) -> dict:
        avg_focal  = self.focal_sum / max(self.n_batches, 1)
        avg_reg    = self.reg_sum   / max(self.n_batches, 1)
        total_loss = avg_focal + 2.0 * avg_reg

        logger.info('=' * 62)
        logger.info(f'  Val focal loss : {avg_focal:.4f}')
        logger.info(f'  Val reg   loss : {avg_reg:.4f}')
        logger.info(f'  Val total loss : {total_loss:.4f}  (focal + 2×reg)')
        logger.info('-' * 62)
        logger.info(f'  {"Class":<14} {"GT objs":>8} {"AvgResp":>9} '
                    f'{"Recall@.3":>10} {"Recall@.5":>10}')
        logger.info('-' * 62)

        results = {}
        for c, name in enumerate(CLASS_NAMES):
            n        = max(self.gt_count[c], 1)
            avg_resp = self.gt_response[c] / n
            rec03    = self.recall_03[c]   / n
            rec05    = self.recall_05[c]   / n
            logger.info(f'  {name:<14} {self.gt_count[c]:>8d} '
                        f'{avg_resp:>9.4f} {rec03:>10.4f} {rec05:>10.4f}')
            results[name] = dict(gt_count=self.gt_count[c],
                                 avg_response=avg_resp,
                                 recall_at_03=rec03,
                                 recall_at_05=rec05)
        logger.info('=' * 62)
        results.update(total_loss=total_loss,
                       focal_loss=avg_focal,
                       reg_loss=avg_reg)

        # Save to CSV
        _save_dir = save_dir or SAVE_DIR
        if dist.get_rank() == 0:
            csv_path = os.path.join(_save_dir, 'val_metrics.csv')
            write_header = not os.path.exists(csv_path)
            with open(csv_path, 'a') as f:
                if write_header:
                    f.write("Epoch,TotalLoss,FocalLoss,RegLoss,"
                            "Car_Recall3,Car_Recall5,Ped_Recall3,Cyc_Recall3\n")
                c_rec3  = results.get('Car',        {}).get('recall_at_03', 0.0)
                c_rec5  = results.get('Car',        {}).get('recall_at_05', 0.0)
                p_rec3  = results.get('Pedestrian', {}).get('recall_at_03', 0.0)
                cy_rec3 = results.get('Cyclist',    {}).get('recall_at_03', 0.0)
                ep_str  = str(epoch) if epoch is not None else '?'
                f.write(f"{ep_str},{total_loss:.4f},{avg_focal:.4f},{avg_reg:.4f},"
                        f"{c_rec3:.4f},{c_rec5:.4f},{p_rec3:.4f},{cy_rec3:.4f}\n")

        return results


def run_validation(checkpoint_path, kitti_root, imageset_dir,
                   batch_size=2, num_workers=4, split='val',
                   rank=0, local_rank=0, world_size=1, device=None):
    """
    Standalone validation runner — called from the training notebook
    or as an independent validation run.
    """
    if device is None:
        device = torch.device(f'cuda:{local_rank}')

    logger = get_logger(rank,
                        log_file=os.path.join(SAVE_DIR, 'val.log'))

    if rank == 0:
        logger.info(f'Validation | {world_size} GPU(s) | ckpt={checkpoint_path}')

    model = BEVWaveFormer(
        pillar_in_ch  = 4,
        pillar_out_ch = 64,
        max_points    = BEV_CFG['max_points_per_pillar'],
        bev_h         = BEV_CFG['bev_h'],
        bev_w         = BEV_CFG['bev_w'],
        stage_dims    = [96, 192, 384, 768],
        depths        = [2, 2, 6, 2],
        use_checkpoint= False,
    )

    ckpt  = torch.load(checkpoint_path, map_location='cpu')
    state = ckpt.get('model_state', ckpt)
    missing, unexpected = model.load_state_dict(state, strict=False)
    if rank == 0:
        if missing:     logger.warning(f'Missing keys   : {missing}')
        if unexpected:  logger.warning(f'Unexpected keys: {unexpected}')
        logger.info(f'Checkpoint epoch={ckpt.get("epoch","?")}, '
                    f'saved_loss={ckpt.get("loss", float("nan")):.4f}')

    model = model.to(device)
    model = DDP(model, device_ids=[local_rank], output_device=local_rank,
                find_unused_parameters=False)
    model.eval()

    split_txt = os.path.join(imageset_dir, f'{split}.txt')
    ds      = KITTIPillarDataset(kitti_root, split, split_txt, BEV_CFG)
    sampler = DistributedSampler(ds, num_replicas=world_size, rank=rank,
                                 shuffle=False, drop_last=False)
    loader  = DataLoader(ds, batch_size=batch_size, sampler=sampler,
                         num_workers=num_workers,
                         collate_fn=kitti_collate_fn, pin_memory=True)

    if rank == 0:
        logger.info(f'Val set: {len(ds)} frames → {len(loader)} batches/rank')

    metrics = ValMetrics(num_classes=3)
    with torch.no_grad():
        for i, batch in enumerate(loader):
            pillars    = batch['pillars'].to(device,    non_blocking=True)
            num_points = batch['num_points'].to(device, non_blocking=True)
            coords     = batch['coords'].to(device,     non_blocking=True)
            bs         = batch['batch_size']

            with autocast('cuda', dtype=torch.bfloat16):
                preds = model(pillars, num_points, coords, bs)

            H_out = preds['heatmap'].shape[-2]
            W_out = preds['heatmap'].shape[-1]
            gt_hm, gt_reg, reg_mask = build_targets_val(
                batch['boxes'], bs, 3, H_out, W_out, BEV_CFG, device)

            metrics.update(preds['heatmap'], gt_hm,
                           preds['reg'],     gt_reg, reg_mask)

            if rank == 0 and (i + 1) % 20 == 0:
                logger.info(f'  {i+1}/{len(loader)} batches...')

    metrics.aggregate(device)
    epoch = ckpt.get('epoch', None)
    if rank == 0:
        metrics.report(logger, epoch=epoch, save_dir=SAVE_DIR)
        logger.info('Validation done.')

    return metrics


print(" validate_checkpoint.py cell loaded.")

 validate_checkpoint.py cell loaded.


In [8]:
# ============================================================
# CELL 7 — Main training entry point
# ============================================================
# Configuration — edit these before running
# ============================================================

CFG = dict(
    kitti_root    = KITTI_ROOT,       # set in Cell 2
    imageset_dir  = IMAGESET_DIR,     # set in Cell 2
    save_dir      = SAVE_DIR,  
    checkpoint_path = CHECKPOINT_INPUT, # set in Cell 2
    resume        = None,             # set to e.g. SAVE_DIR+'/session_checkpoint.pth' to resume
    epochs        = 160,
    batch_size    = 23,                # per-rank; with 1 GPU → global=2
    lr            = 1e-4,
    weight_decay  = 0.01,
    num_workers   = 4,
    log_interval  = 50,
    save_interval = 5,                # save checkpoint every N epochs
    session_hours = 11.0,             # save-and-exit after this many wall-clock hours
)

# ── Auto-detect whether we're resuming from a session checkpoint ──────────
_session_ckpt = os.path.join(CFG['checkpoint_path'], 'session_checkpoint.pth')
if CFG['resume'] is None and os.path.exists(_session_ckpt):
    print(f"  Found session_checkpoint.pth — auto-resuming from {_session_ckpt}")
    CFG['resume'] = _session_ckpt

# ── DDP init (single-process for Kaggle single-GPU; works for multi-GPU too) ─
import torch.distributed as dist_mod

if not dist_mod.is_initialized():
    os.environ.setdefault('MASTER_ADDR', 'localhost')
    os.environ.setdefault('MASTER_PORT', '29500')
    os.environ.setdefault('RANK',        '0')
    os.environ.setdefault('WORLD_SIZE',  '1')
    os.environ.setdefault('LOCAL_RANK',  '0')
    dist_mod.init_process_group(backend='nccl', init_method='env://')

rank       = dist_mod.get_rank()
world_size = dist_mod.get_world_size()
local_rank = int(os.environ.get('LOCAL_RANK', 0))

torch.cuda.set_device(local_rank)
device = torch.device(f'cuda:{local_rank}')

os.makedirs(CFG['save_dir'], exist_ok=True)
logger = get_logger(rank, log_file=os.path.join(CFG['save_dir'], 'train.log'))

if rank == 0:
    logger.info(f'World size     : {world_size}  |  Device : {device}')
    logger.info(f'Per-rank batch : {CFG["batch_size"]}  |  '
                f'Global batch   : {CFG["batch_size"] * world_size}')
    logger.info(f'AMP dtype      : bfloat16  (RTX Pro 6000 Ada Lovelace)')
    logger.info(f'Session limit  : {CFG["session_hours"]} hours')

# ── Model ──────────────────────────────────────────────────────────────────
model = BEVWaveFormer(
    pillar_in_ch  = 4,
    pillar_out_ch = 64,
    max_points    = BEV_CFG['max_points_per_pillar'],
    bev_h         = BEV_CFG['bev_h'],
    bev_w         = BEV_CFG['bev_w'],
    stage_dims    = [96, 192, 384, 768],
    depths        = [2, 2, 6, 2],
    use_checkpoint= True,
).to(device)

model = DDP(model,
            device_ids=[local_rank],
            output_device=local_rank,
            find_unused_parameters=False,
            gradient_as_bucket_view=True)

if rank == 0:
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    logger.info(f'Model parameters: {n_params:.1f} M')

# ── Optimizer + Scheduler ──────────────────────────────────────────────────
spectral_params = [p for n, p in model.named_parameters()
                   if 'log_alpha' in n or 'log_v' in n]
other_params    = [p for n, p in model.named_parameters()
                   if 'log_alpha' not in n and 'log_v' not in n]

optimizer = torch.optim.AdamW([
    {'params': other_params,    'lr': CFG['lr']},
    {'params': spectral_params, 'lr': CFG['lr'] * 0.1,
     'weight_decay': 0.0},
], weight_decay=CFG['weight_decay'])

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CFG['epochs'], eta_min=CFG['lr'] * 1e-2)

criterion = BEVLoss(reg_weight=2.0).to(device)

# ── Data ───────────────────────────────────────────────────────────────────
train_loader, val_loader, train_sampler = build_dataloaders(
    CFG['kitti_root'], CFG['imageset_dir'],
    batch_size=CFG['batch_size'],
    num_workers=CFG['num_workers'],
    rank=rank, world_size=world_size,
)

# ── Resume ─────────────────────────────────────────────────────────────────
start_epoch = 0
if CFG['resume']:
    if os.path.exists(CFG['resume']):
        ckpt = torch.load(CFG['resume'], map_location='cpu')
        model.module.load_state_dict(ckpt['model_state'])
        model.module.to(device)
        optimizer.load_state_dict(ckpt['optimizer_state'])
        # Restore scheduler state if present (session checkpoints include it)
        if 'scheduler_state' in ckpt:
            scheduler.load_state_dict(ckpt['scheduler_state'])
        start_epoch = ckpt['epoch'] + 1
        logger.info(f'Resumed from epoch {ckpt["epoch"]}, '
                    f'loss={ckpt.get("loss", float("nan")):.4f}')
        if dist_mod.is_initialized():
            dist_mod.barrier()
    else:
        logger.warning(f'Resume path not found: {CFG["resume"]} — starting fresh.')

# ── Session timer ──────────────────────────────────────────────────────────
session_timer = SessionTimer(limit_hours=CFG['session_hours'])
val_loss      = float('inf')
best_val_loss = float('inf')

if rank == 0:
    logger.info(f'Starting training from epoch {start_epoch} '
                f'to epoch {CFG["epochs"]-1}')


import torch, gc
torch.cuda.empty_cache(); gc.collect()
allocated = torch.cuda.memory_allocated(0) / 1024**3
reserved  = torch.cuda.memory_reserved(0)  / 1024**3
total     = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU        : {torch.cuda.get_device_name(0)}")
print(f"Total VRAM : {total:.2f} GB")
print(f"Allocated  : {allocated:.2f} GB")
print(f"Reserved   : {reserved:.2f} GB")
print(f"Free       : {total - reserved:.2f} GB")



# ── Training loop ──────────────────────────────────────────────────────────
for epoch in range(start_epoch, CFG['epochs']):
    train_sampler.set_epoch(epoch)

    train_loss = train_one_epoch(
        model, train_loader, optimizer, criterion,
        device, epoch, logger, CFG['log_interval'])

    val_loss = validate(
        model, val_loader, criterion, device, epoch, logger)

    scheduler.step()

    logger.info(
        f'─── Epoch {epoch:04d} | train={train_loss:.4f} '
        f'| val={val_loss:.4f} '
        f'| lr={scheduler.get_last_lr()[0]:.2e} '
        f'| session_remaining={session_timer.remaining_str()} ───')

    # Periodic checkpoint
    if (epoch + 1) % CFG['save_interval'] == 0:
        save_checkpoint(model, optimizer, epoch, val_loss,
                        CFG['save_dir'], rank)

    # Best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(model, optimizer, epoch, val_loss,
                        CFG['save_dir'], rank, filename='best.pth')
        logger.info(f'  ↳ New best val loss: {val_loss:.4f}')

    # ── 11-hour session save-and-exit ──────────────────────────────────────
    if session_timer.should_save_and_exit():
        logger.info(
            f' Session time limit reached after epoch {epoch}. '
            f'Saving session checkpoint and stopping.')
        save_session_checkpoint(
            model, optimizer, scheduler, epoch,
            val_loss, CFG['save_dir'], rank, logger)
        # Also save a regular epoch checkpoint
        save_checkpoint(model, optimizer, epoch, val_loss,
                        CFG['save_dir'], rank,
                        filename=f'ckpt_epoch{epoch:04d}_session_end.pth')
        if dist_mod.is_initialized():
            dist_mod.barrier()
        break

    if dist_mod.is_initialized():
        dist_mod.barrier()

# ── Final checkpoint ────────────────────────────────────────────────────────
else:
    # Only reached if loop completed without break (all epochs done)
    save_checkpoint(model, optimizer, CFG['epochs'] - 1, val_loss,
                    CFG['save_dir'], rank, filename='final.pth')
    logger.info(' Training complete — all epochs finished.')

if dist_mod.is_initialized():
    dist_mod.destroy_process_group()
    print(" Process group destroyed.")

  Found session_checkpoint.pth — auto-resuming from /kaggle/input/datasets/codingyodha/session-1-checkpoints/session_checkpoint.pth


[2026-03-31 06:07:03][INFO] World size     : 1  |  Device : cuda:0
[2026-03-31 06:07:03][INFO] Per-rank batch : 23  |  Global batch   : 23
[2026-03-31 06:07:03][INFO] AMP dtype      : bfloat16  (RTX Pro 6000 Ada Lovelace)
[2026-03-31 06:07:03][INFO] Session limit  : 11.0 hours
[2026-03-31 06:07:07][INFO] Model parameters: 23.5 M
[2026-03-31 06:07:07][INFO] Resumed from epoch 52, loss=1.6436
/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[2026-03-31 06:07:07][INFO] Starting training from epoch 53 to epoch 159


GPU        : NVIDIA RTX PRO 6000 Blackwell Server Edition
Total VRAM : 94.97 GB
Allocated  : 0.35 GB
Reserved   : 0.38 GB
Free       : 94.59 GB


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:865: UserWarning: Grad strides do not match bucket view strides. This may indicate grad was not created according to the gradient layout contract, or that the param's strides changed since DDP was constructed.  This is not an error, but may impair performance.
grad.sizes() = [768, 768, 1, 1], strides() = [768, 1, 768, 768]
bucket_view.sizes() = [768, 768, 1, 1], strides() = [768, 1, 1, 1] (Triggered internally at /pytorch/torch/csrc/distributed/c10d/reducer.cpp:330.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
[2026-03-31 06:07:18][INFO] Epoch 0053 | Step 00000/00162 | loss=1.2487 | grad_norm=138.972 | max_norm=7.000 | 10.9s elapsed
[2026-03-31 06:10:23][INFO] Epoch 0053 | Step 00050/00162 | loss=1.3433 | grad_norm=159.338 | max_norm=7.000 | 195.2s elapsed
[2026-03-31 06:13:27][INFO] Epoch 0053 | Step 00100/00162 | loss=1.5456 | grad_norm=125.093 | max_norm=7.000 

 Process group destroyed.


In [9]:
# # ============================================================
# # CELL 8 — Run standalone validation on any saved checkpoint
# # ============================================================
# # Run this cell independently of Cell 7.
# # Make sure Cells 2–6 have been executed first (definitions loaded).
# # ──────────────────────────────────────────────────────────────

# CKPT_TO_VALIDATE = os.path.join(SAVE_DIR, 'best.pth')   # ← change as needed

# if not os.path.exists(CKPT_TO_VALIDATE):
#     print(f" Checkpoint not found: {CKPT_TO_VALIDATE}")
# else:
#     # Re-init DDP if the process group was destroyed after training
#     if not dist.is_initialized():
#         os.environ.setdefault('MASTER_ADDR', 'localhost')
#         os.environ.setdefault('MASTER_PORT', '29501')   # different port
#         os.environ.setdefault('RANK',        '0')
#         os.environ.setdefault('WORLD_SIZE',  '1')
#         os.environ.setdefault('LOCAL_RANK',  '0')
#         dist.init_process_group(backend='nccl', init_method='env://')

#     _rank       = dist.get_rank()
#     _local_rank = int(os.environ.get('LOCAL_RANK', 0))
#     _world_size = dist.get_world_size()
#     _device     = torch.device(f'cuda:{_local_rank}')
#     torch.cuda.set_device(_local_rank)

#     run_validation(
#         checkpoint_path = CKPT_TO_VALIDATE,
#         kitti_root      = KITTI_ROOT,
#         imageset_dir    = IMAGESET_DIR,
#         batch_size      = 2,
#         num_workers     = 4,
#         split           = 'val',
#         rank            = _rank,
#         local_rank      = _local_rank,
#         world_size      = _world_size,
#         device          = _device,
#     )

#     dist.destroy_process_group()
#     print(" Validation complete.")